<a href="https://colab.research.google.com/github/ezeljko1981/Quantum_Computer/blob/main/bv_2q%26readout_batch_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.noise import depolarizing_error
from qiskit_aer.noise import ReadoutError
import time
from datetime import date, datetime

def bernstein_vazirani(secret_string):
    n = len(secret_string)
    circuit = QuantumCircuit(n + 1, n)
    # --- Step 1: Initialize Auxiliary Qubit to |-> state ---
    circuit.x(n)      # Flip to |1>
    circuit.barrier()
    circuit.h(n)      # Apply Hadamard to get |->
    # --- Step 2: Apply Hadamard gates to input qubits ---
    circuit.h(range(n))
    circuit.barrier()
    # --- Step 3: The Quantum Oracle ---
    # For every '1' in the secret string, apply a CNOT gate
    # where the input qubit is the control and the ancilla is the target
    for i, bit in enumerate(reversed(secret_string)): #1100 -> 0011
        if bit == '1':
            circuit.cx(i, n)
    circuit.barrier()
    # --- Step 4: Apply Hadamard gates to input qubits again ---
    circuit.h(range(n))
    # --- Step 5: Measure the input qubits ---
    circuit.measure(range(n), range(n))
    return circuit


def bernstein_vazirani_batch(err_value_2q, err_readout_matrix):
  secrets = ["00", "10", "11",
           "0000", "1010", "1111",
           "00000000", "10101010", "11111111",
           "0000000000000000", "1010101010101010","1111111111111111",
           "00000000000000000000000000000000", "10101010101010101010101010101010", "11111111111111111111111111111111",
           "0000000000000000000000000000000000000000000000000000000000000000", "1010101010101010101010101010101010101010101010101010101010101010", "1111111111111111111111111111111111111111111111111111111111111111",
           "00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000", "10101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010", "11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111",
           "0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000", "1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010", "1111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111",
           "00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000", "10101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010", "11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111",
           "0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000", "1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010", "1111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111"
          ]
  print("Time[sec],CountBestOutputValue,BestOutputValue")
  for s in secrets:
    bv_circuit = bernstein_vazirani(s)
    noise_model = NoiseModel()
    # 2Q Error
    p_2q = err_value_2q
    error_2q = depolarizing_error(p_2q, 2) # 2 označava dvo-kubitnu grešku
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])
    # Readout Error
    p_readout = err_readout_matrix
    error_ro = ReadoutError(p_readout)
    noise_model.add_all_qubit_readout_error(error_ro)
    # Simulation
    simulator = AerSimulator(noise_model=noise_model)
    simulator.set_max_qubits(1025)
    compiled_circuit = transpile(bv_circuit, simulator)
    shoots = 1024
    start_time = time.perf_counter()
    sim_result = simulator.run(compiled_circuit, shoots = shoots).result()
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    counts = sim_result.get_counts()
    best_state = max(counts, key=counts.get)
    best_count = counts[best_state]
    print(f"{execution_time},{best_count},{best_state}")
  return f"PARAMS: DateTime: {datetime.now()} 2Q: {err_value_2q} Readout Matrix [P(0|0), P(1|0)] i [P(0|1), P(1|1)] : {err_readout_matrix}"

# Define 2Q Error and Readout Error Matrix: [P(0|0), P(1|0)] i [P(0|1), P(1|1)]
bernstein_vazirani_batch(0.00354, [[0.99487, 0.00513], [0.00513, 0.99487]]) # ibm_boston
# No Errors case:
# bernstein_vazirani_batch(0, [[1, 0], [0, 1]])
# bernstein_vazirani_batch(0.00109, [[0.99487, 0.00513], [0.00513, 0.99487]])